# PNG/BMP — Colab T4 erişim ve ortam kontrolü

Bu ayrı notebook yalnız **tek bir T4 tahsisi** içinde donanım/paket ve kaynak metadatasını okur.
Çalışma zamanı: **Python 3 / T4 GPU / Runtime Version 2026.07**.
Kod için toplam 60 saniye; `nvidia-smi` için 10 saniye /8 KiB çıktı sınırı var.
Torch import edilmez; model, tensor işlemi ve optimizer oluşturulmaz.
Önceki kapasite ölçümleri ve yerel GPU işleri yeniden başlatılmaz.

Hücreyi bir kez çalıştırın. Sonucu kaydedip GPU oturumunu kapatın.
Bu bir eğitim veya performans ölçümü değildir.


In [ ]:
import hashlib
import importlib.metadata
import json
import os
import platform
import resource
import shutil
import stat
import subprocess
import tempfile
import time
from pathlib import Path, PurePosixPath

MAX_SECONDS = 60
OUTPUT_LIMIT = 65536
EXPECTED = {
    "python": "3.12.13", "numpy": "2.0.2", "Pillow": "11.3.0",
    "torch": "2.11.0+cu128",
}
OUT = Path("/content/png-bmp-colab-gpu-inventory-20261106")


def read_bounded(path, limit):
    if any(p.is_symlink() for p in (path, *path.parents)):
        raise ValueError("metadata symlink")
    fd = os.open(path, os.O_RDONLY | os.O_NOFOLLOW | os.O_NONBLOCK)
    with os.fdopen(fd, "rb") as stream:
        if not stat.S_ISREG(os.fstat(stream.fileno()).st_mode):
            raise ValueError("metadata file type")
        raw = stream.read(limit + 1)
    if len(raw) > limit:
        raise ValueError("metadata byte limit")
    return raw


def write_record(name, value):
    raw = (json.dumps(value, indent=2, sort_keys=True) + "\n").encode()
    if len(raw) > 16384:
        raise ValueError("receipt byte limit")
    fd = os.open(OUT / name, os.O_WRONLY | os.O_CREAT | os.O_EXCL | os.O_NOFOLLOW, 0o600)
    with os.fdopen(fd, "wb") as stream:
        stream.write(raw)
        stream.flush()
        os.fsync(stream.fileno())
    return hashlib.sha256(raw).hexdigest()


def cgroup_metadata():
    raw = read_bounded(Path("/proc") / str(os.getpid()) / "cgroup", 4096).decode()
    candidates = []
    for line in raw.splitlines():
        _, controllers, relative = line.split(":", 2)
        path = PurePosixPath(relative)
        if not path.is_absolute() or ".." in path.parts or len(path.parts) > 32:
            raise ValueError("cgroup identity")
        if line.startswith("0::"):
            candidates.append(("v2", Path("/sys/fs/cgroup").joinpath(*path.parts[1:]), {
                "memory_limit_bytes": "memory.max", "swap_limit_bytes": "memory.swap.max",
                "memory_peak_bytes": "memory.peak", "memory_current_bytes": "memory.current",
            }))
        elif "memory" in controllers.split(","):
            candidates.append(("v1", Path("/sys/fs/cgroup/memory").joinpath(*path.parts[1:]), {
                "memory_limit_bytes": "memory.limit_in_bytes",
                "memory_and_swap_limit_bytes": "memory.memsw.limit_in_bytes",
                "memory_peak_bytes": "memory.max_usage_in_bytes",
                "memory_current_bytes": "memory.usage_in_bytes",
            }))
    observations = []
    for kind, folder, names in candidates[:2]:
        counters = {}
        for key, name in names.items():
            try:
                text = read_bounded(folder / name, 32).decode().strip()
                if text == "max":
                    counters[key] = "unbounded"
                elif text.isascii() and text.isdecimal() and len(text) <= 20:
                    number = int(text)
                    counters[key] = number if number < 2**60 else "unbounded"
                else:
                    counters[key] = "unavailable"
            except (OSError, UnicodeError, ValueError):
                counters[key] = "unavailable"
        observations.append({"version": kind, "counters": counters})
    return {"controllers": observations, "delegated_enforcement": "unverified"}


def gpu_metadata(remaining):
    limiter = shutil.which("prlimit")
    binary = shutil.which("nvidia-smi")
    if limiter is None or binary is None:
        return {"status": "unavailable_required_inventory_tool"}
    arguments = [limiter, "--fsize=8192:8192", "--cpu=5:5", "--", binary,
                 "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"]
    with tempfile.TemporaryFile(dir=OUT) as capture:
        try:
            process = subprocess.run(  # noqa: S603 -- fixed system-tool arguments
                arguments, stdout=capture, stderr=subprocess.STDOUT,
                                     timeout=min(10, remaining), check=False)
        except subprocess.TimeoutExpired:
            return {"status": "unavailable_inventory_timeout"}
        capture.seek(0)
        raw = capture.read(8193)
    if process.returncode != 0 or len(raw) > 8192:
        return {"status": "unavailable_inventory_command", "returncode": process.returncode}
    lines = raw.decode("ascii").strip().splitlines()
    if len(lines) != 1:
        return {"status": "unavailable_single_gpu_required"}
    fields = [s.strip() for s in lines[0].split(",")]
    if (len(fields) != 3 or not 0 < len(fields[0]) <= 128
            or not fields[1].isdigit() or not 0 < int(fields[1]) <= 524288
            or len(fields[2]) > 32 or not fields[2].replace(".", "").isdigit()):
        return {"status": "unavailable_inventory_format"}
    return {"status": "visible_metadata_only", "name": fields[0],
            "vram_mib": int(fields[1]), "driver": fields[2]}


def inspect_inventory():
    started = time.monotonic()
    OUT.mkdir(mode=0o700, exist_ok=False)
    write_record("attempt.json", {"schema": "colab-gpu-inventory-attempt-v1",
                                 "max_seconds": MAX_SECONDS, "retry": False})
    result = {"schema": "colab-gpu-inventory-v1", "runtime_version_selected": "2026.07",
              "python": platform.python_version(), "packages": {}, "models_created": 0,
              "optimizer_updates": 0, "tensor_operations": 0, "torch_imported": False,
              "resource_enforcement": "unverified", "profile_attempt_started": False}
    try:
        for package in ("numpy", "Pillow", "torch", "onnx", "onnxruntime"):
            try:
                result["packages"][package] = importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError:
                result["packages"][package] = "unavailable"
        observed = {"python": result["python"], **result["packages"]}
        result["package_metadata_matches"] = all(observed.get(k) == v for k, v in EXPECTED.items())
        result["cgroup"] = cgroup_metadata()
        result["process_limits"] = {
            name: list(resource.getrlimit(key))
            for name, key in (("address_space", resource.RLIMIT_AS),
                              ("cpu_seconds", resource.RLIMIT_CPU),
                              ("file_bytes", resource.RLIMIT_FSIZE))
        }
        remaining = MAX_SECONDS - (time.monotonic() - started)
        if remaining <= 0:
            raise TimeoutError("inventory deadline")
        result["gpu"] = gpu_metadata(remaining)
        result["status"] = ("gpu_metadata_ready_only"
                            if result["package_metadata_matches"]
                            and result["gpu"]["status"] == "visible_metadata_only"
                            else "unavailable_gpu_or_environment")
    except (OSError, UnicodeError, ValueError, TimeoutError) as error:
        result["status"] = "unavailable_inventory"
        result["failure_type"] = type(error).__name__
    result["seconds"] = time.monotonic() - started
    if result["seconds"] > MAX_SECONDS:
        result["status"] = "unavailable_deadline"
    result["receipt_sha256"] = write_record("result.json", result)
    output_bytes = sum(p.stat().st_size for p in OUT.iterdir() if p.is_file())
    if output_bytes > OUTPUT_LIMIT:
        raise ValueError("inventory output limit")
    print(json.dumps(result, indent=2, sort_keys=True))


inspect_inventory()
